# XD2-Net — 02: Training

Trains one XD2-Net model per prefix-length bucket using saved artifacts
from `01_data_preparation.ipynb`. Saves checkpoints and loss curves.

**Prerequisite:** run notebook 01 first.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/xd2-net'
    %pip install -e {REPO_ROOT} --quiet
else:
    REPO_ROOT = os.path.abspath('..')
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
import pickle
import logging
import matplotlib.pyplot as plt
import pandas as pd

from xd2net.config import load_config
from xd2net.model.xd2net import XD2Net
from xd2net.data.dataset import make_dataloaders
from xd2net.training.trainer import Trainer

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s',
                    datefmt='%H:%M:%S')

cfg = load_config('configs/bpic2012.yaml')
artifacts_dir = os.path.join(cfg.output.workspace_dir, 'artifacts')
ckpt_dir = os.path.join(cfg.output.workspace_dir, 'checkpoints')

print('Artifacts dir:', artifacts_dir)

In [ ]:
# ── Optional: train only a subset of buckets ─────────────────────────────────
# Set to None to train all buckets, or e.g. ['1_10', '1_15'] for a quick test
BUCKETS_TO_TRAIN = None  # e.g. ['1_10', '1_15']

In [ ]:
results_rows = []
histories = {}

for n_size in cfg.data.prefix_lengths:
    bucket_key = f'1_{n_size}'
    if BUCKETS_TO_TRAIN and bucket_key not in BUCKETS_TO_TRAIN:
        continue

    art_path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    if not os.path.exists(art_path):
        print(f'{bucket_key}: artifact not found, skipping')
        continue

    with open(art_path, 'rb') as f:
        art = pickle.load(f)

    print(f'\n{"="*60}\nBucket {bucket_key}')

    model = XD2Net(
        dim_dynamic=art['dim_dynamic'],
        dim_static=art['dim_static'],
        exp_dim=art['exp_dim'],
        max_len=art['max_len'],
        cfg=cfg.model,
    )
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

    train_loader, val_loader = make_dataloaders(
        art['inputs_train'], art['outputs_train'], art['lengths_train'],
        val_split=cfg.training.val_split,
        batch_size=cfg.training.batch_size,
    )

    trainer = Trainer(model, cfg.training, ckpt_dir, bucket=bucket_key)
    history = trainer.train(train_loader, val_loader)
    histories[bucket_key] = history

    results_rows.append({'bucket': bucket_key, 'best_val_loss': min(history['val_loss']),
                         'best_val_auc': max(history['val_auc']), 'epochs': len(history['val_loss'])})

print('\nTraining complete.')

In [ ]:
# ── Plot loss curves ──────────────────────────────────────────────────────────
for bucket_key, h in histories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h['train_loss'], label='train')
    ax1.plot(h['val_loss'],   label='val')
    ax1.set_title(f'{bucket_key} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss'); ax1.legend()

    ax2.plot(h['val_auc'])
    ax2.set_title(f'{bucket_key} — Val AUC')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('ROC-AUC')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
df_summary = pd.DataFrame(results_rows)
print(df_summary.to_string(index=False))